In [1]:
#from src.acquisition import *

from pathlib import Path
root = Path.cwd()

import pandas as pd

from osgeo import gdal

# GDAL configurations used to successfully access LP DAAC Cloud Assets via vsicurl 
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE','~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR', '~/cookies.txt')
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN','EMPTY_DIR')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS','TIF')
gdal.SetConfigOption('GDAL_HTTP_UNSAFESSL', 'YES')
gdal.SetConfigOption('GDAL_HTTP_MAX_RETRY', '10')
gdal.SetConfigOption('GDAL_HTTP_RETRY_DELAY', '0.5')


import earthaccess
earthaccess.login(strategy='netrc')


import stackstac
import geopandas as gpd
import numpy as np
import xarray as xr
import netCDF4
import rioxarray
from rasterio.enums import Resampling
from datetime import datetime
import matplotlib.pyplot as plt
from functools import reduce


from pystac_client import Client

from rasterio.features import rasterize


In [ ]:
class Site():
    def __init__(self,root,site_name,epsg):
        self.site_name = site_name
        self.root = root
        self.boundary = gpd.read_file(self.root / 'data' / self.site_name / f'{self.site_name}_boundary.gpkg')
        self.epsg = epsg

        assert self.boundary.crs.to_epsg() == self.epsg, (
            f"Boundary should have epsg:{self.epsg}, but has epsg:{self.boundary.crs.to_epsg()}"
        )

        self.bbox_utm, self.bbox_4326 = self.get_bbox()
        
        
    @property
    def tree_canopy(self):
        t = rioxarray.open_rasterio(self.root / 'data' / self.site_name / f'{self.site_name}_tree_canopy.tif')
        t = t.rio.reproject(f"EPSG:{self.epsg}")
        t = t.rio.clip(geometries=self.boundary.geometry)
        return t
    
    # add property methods for nlcd?
    
    def get_bbox(self):
        boundary_4326 = self.boundary.to_crs(epsg=4326)
        bbox_utm = tuple(self.boundary.total_bounds)
        bbox_4326 = tuple(boundary_4326.total_bounds)
        return bbox_utm, bbox_4326
    

class HlsDownloader():
    def __init__(self,site_object,year):
        self.landsat_bands = ["B02","B03","B04","B05","B06","B07","B10","B11","Fmask"]
        self.sentinel_bands = ["B02","B03","B04","B05","B06","B07","B8A","B11","B12","Fmask"]
        self.site = site_object
        self.year = year

        self.all_landsat = None
        self.all_sentinel = None

        self.filtered_landsat = None
        self.filtered_sentinel = None

        self.pixel_threshold = None

        self.landsat_timestamps_df = None
        self.sentinel_timestamps_df = None

        self.landsat_months_to_keep = None
        self.sentinel_months_to_keep = None
       
            
    def get_stac_items(self):

        catalog = Client.open("https://cmr.earthdata.nasa.gov/stac/LPCLOUD")


        query_results = catalog.search(
            bbox=self.site.bbox_4326,
            collections=["HLSL30.v2.0","HLSS30.v2.0"],
            datetime=f"{self.year}-04-01/{self.year}-11-30"
        ).item_collection()


        landsat_items = [item for item in query_results if item.collection_id == "HLSL30_2.0"]
        print(f'found {len(landsat_items)} landsat images')
        sentinel_items = [item for item in query_results if item.collection_id == "HLSS30_2.0"]
        print(f'found {len(sentinel_items)} sentinel images')

        # convert to xarray and crop to city boundaries
        self.all_landsat = self.process_items(landsat_items,self.landsat_bands)
        self.all_sentinel = self.process_items(sentinel_items,self.sentinel_bands)

        self.cloud_mask = self.make_cloud_mask(self.all_landsat)
        

        # get pixel threshold
        if self.pixel_threshold == None:
            self.pixel_threshold = self.get_valid_pixel_threshold(data=self.all_landsat)
            

        # remove invalid timesteps, also mask and scale data
        self.filtered_landsat = self.filter_mask_scale('landsat')
        self.filtered_sentinel = self.filter_mask_scale('sentinel')

        self.landsat_timestamps_df = self.make_timestamps_df('landsat')
        self.sentinel_timestamps_df = self.make_timestamps_df('sentinel')

    def process_items(self,items,bands):

        # create xarray
        x = stackstac.stack(
            items,
            epsg=self.site.epsg,
            resolution=30,
            bounds=self.site.bbox_utm,
            assets=bands).where(lambda x: x > 0, other=np.nan)
        # filter for images with low cloud cover
        lowcloud = x[x["eo:cloud_cover"] < 20]

        return lowcloud.rio.clip(geometries=self.site.boundary.geometry)

    def make_cloud_mask(self,satellite):

        fmask_values= np.unique(satellite.sel(band='Fmask'))
        # convert fmask values to binary
        all_bits = list()
        for v in fmask_values[:-1]:
                all_bits.append(format(int(v), 'b').zfill(8))

        # cloud = bit index 1, cloud shadow = bit index, snow/ice = bit index 4
        qa = list()
        for b in all_bits:
                if int(b[3]) + int(b[4]) + int(b[6]) == 0:    # if bits in these positions are all zero then quality is good
                        qa.append('good')
                else:
                        qa.append('poor')

        keep_values = list()
        for (quality, value) in (zip(qa,fmask_values)):
                if quality == 'good':
                        keep_values.append(value)

        return keep_values

    def get_valid_pixel_threshold(self,data):
        s = data.isel(band=0,time=0)

        rasterized = rasterize(self.site.boundary.geometry,
                                        out_shape = s.shape,
                                        fill = 0,
                                        out = None,
                                        transform = s.rio.transform(),
                                        all_touched = False,
                                        default_value = 1,
                                        dtype = 'int8')
        return rasterized.sum()


    def filter_mask_scale(self,satellite):
        data = getattr(self,f'all_{satellite}')

        ################ apply qa mask to data #######################
        fmask = data.sel(band='Fmask')
        # number of bands
        assert data.dims[1] == 'band', f"array dimensions should be (time,band,y,x),not {data.dims}"
        dim_size = data.shape[1]
        # reshape fmask band 
        fmask = np.repeat(fmask.values[:, np.newaxis, :,:], dim_size, axis=1)
        # apply fmask to satellite data
        masked = data.where(np.isin(fmask,self.cloud_mask))
        masked = masked.drop_sel(band='Fmask')  # drop fmask band

        ############## filter out timesteps with incomplete data ############
        num_valid_pixels = masked.isel(band=1).count(dim=('y','x'))

        # make boolean arrays showing if timesteps at least 70% of pixels
        valid_slices = num_valid_pixels.values >= self.pixel_threshold*.7

        # get indices of valid timesteps
        keep_index = []
        for i,a in enumerate(valid_slices):
            if a == True:
                keep_index.append(i)

        filtered = masked.isel(time=keep_index)


        ################ apply scaling factors ####################
        if satellite == 'landsat':
                scaled1 = filtered.isel(band=slice(0,6)) * .0001   
                scaled2 = filtered.isel(band=slice(6,None)) * .01       

                scaled = xr.concat([scaled1,scaled2], dim='band')
        else:
                scaled = filtered * .0001
                

        return scaled
    
    def make_timestamps_df(self, satellite):
        all_data = getattr(self, f'all_{satellite}')
        filtered_data = getattr(self, f'filtered_{satellite}')

        all_dates = pd.to_datetime(all_data.time.values).strftime('%Y-%m-%d')
        filtered_dates = pd.to_datetime(filtered_data.time.values).strftime('%Y-%m-%d')
        print(f'Suitable {satellite} images: {filtered_dates}')

        # add nans to make sure length matches
        filtered_dates = list(filtered_dates) + [np.nan] * (len(all_dates) - len(filtered_dates))

        return pd.DataFrame({f'all_{satellite}': all_dates, f'filtered_{satellite}': filtered_dates})
    
    def get_months_to_keep(self,satellite,list_of_months):
        sat = getattr(self,f'filtered_{satellite}')

        d = pd.to_datetime(sat.time.values) 
        d_filtered = d[d.month.isin(list_of_months)].values

        setattr(self, f'{satellite}_months_to_keep', d_filtered)

        path = self.site.root / 'data' / self.site.site_name / 'satellite_image_dates.txt'
        path.touch()  
        with path.open('a+') as file:
            file.write(f'\nSite: {self.site.site_name} \nSatellite: {satellite}\n Dates: {d_filtered}\n\n' )
    
    def plot_data(self,satellite,type):
        a = getattr(self,f'{type}_{satellite}')
        a.isel(band=0).plot(col='time',col_wrap=5)


class DataCombiner():
    def __init__(self,site_obj,data_objs,year):
        self.site_obj = site_obj
        self.data_objs = data_objs
        self.year = year

        self.output_bands = None
        self.output_indices = None


    def combine_data_by_satellite(self,satellite):
        # takes data from same satellite and different years and creates unified time series
        if satellite == 'sentinel':
            band_names = ['blue','green','red','rededge1','rededge2','rededge3','nir','sw1','sw2']
        else:
            band_names = ['blue','green', 'red','nir','sw1','sw2','tir1','tir2']

        data_list = []
        for obj in self.data_objs:
            months_to_keep = getattr(obj,f'{satellite}_months_to_keep')
            filtered_data = getattr(obj,f'filtered_{satellite}')
            if months_to_keep is not None:
                f = filtered_data.sel(time=months_to_keep)
                dt = pd.to_datetime(f.time).floor('d')
                dt = dt.map(lambda x: x.replace(year=self.year)).to_numpy() # set to common year
                f = f.assign_coords({'time':dt})
                data_list.append(f)
        # merge all years into one time series
        data = reduce(lambda x, y: x.combine_first(y), data_list)
        # rename bands
        data = data.assign_coords({'band':band_names})
            
        return data
    
    def aggregate_annual_monthly(self,data):
        # aggregates time series data to annual and  monthly time steps
        annual = data.median('time').expand_dims('time').assign_coords({'time':['annual']})
        monthly = data.groupby('time.month').median('time').assign_coords({'month':['april','july','october']}).rename({'month':'time'})
        return annual, monthly
    
    def combine_data(self):
        # create unified time series for each satellite
        landsat_data = self.combine_data_by_satellite(satellite='landsat')
        sentinel_data = self.combine_data_by_satellite(satellite='sentinel')

        # names of shared bands, unique sentinel bands & unique landsat bands
        shared_band_list = ['blue','green','red','nir','sw1','sw2']
        sentinel_band_list = ['rededge1', 'rededge2', 'rededge3']
        landsat_band_list = ['tir1','tir2']

        # lists to hold aggregated data
        annual_list = []
        monthly_list = []

        shared_bands = landsat_data.sel(band=shared_band_list).combine_first(sentinel_data.sel(band=shared_band_list))
        l_only_bands = landsat_data.sel(band=landsat_band_list)
        s_only_bands = sentinel_data.sel(band=sentinel_band_list)

        for a in [shared_bands,l_only_bands,s_only_bands]:
            b = a.median('time').expand_dims('time').assign_coords({'time':['annual']})
            annual_list.append(b)

        for a in [shared_bands,l_only_bands,s_only_bands]:
            b = a.groupby('time.month').median('time').assign_coords({'month':['april','july','october']}).rename({'month':'time'})
            monthly_list.append(b)
            
        # stack all aggregated bands together
        annual = xr.concat(annual_list,dim='band')
        monthly = xr.concat(monthly_list,dim='band')

        # align tree canopy raster with satellite data; bilinear resampling strategy
        tc = self.site_obj.tree_canopy.assign_coords({'band':['tc']})
        tc_align = tc.rio.reproject_match(annual.isel(band=1,time=0), resampling=Resampling.bilinear).expand_dims({'time':['annual']})

        c1 = xr.concat([annual,tc_align],dim='band')

        self.output_bands = xr.concat([c1,monthly],dim='time')

        ###### calculate indices #######
        # daily indices calculated first, then aggregated to month and year.

        shared_indices = []
        # tdvi
        shared_indices.append(1.5*((shared_bands.sel(band='nir')-shared_bands.sel(band='red'))/(shared_bands.sel(band='nir')**2+shared_bands.sel(band='red')+0.5)**0.5))
        # ndwi
        shared_indices.append((shared_bands.sel(band='green')-shared_bands.sel(band='nir'))/(shared_bands.sel(band='green')+shared_bands.sel(band='nir')))
        # msavi
        shared_indices.append(((2*shared_bands.sel(band='nir')+1)-((2*shared_bands.sel(band='nir')+1)**2-8*(shared_bands.sel(band='nir')-shared_bands.sel(band='red')))**0.5)/2)
        #brightness
        shared_indices.append((shared_bands.sel(band='blue')*0.3029)+(shared_bands.sel(band='green')*0.2786)+(shared_bands.sel(band='red')*0.4733)+(shared_bands.sel(band='nir')*0.5599)+(shared_bands.sel(band='sw1')*0.508)+(shared_bands.sel(band='sw2')*0.1872))
        # greenness
        shared_indices.append((shared_bands.sel(band='blue')*(-0.2941))+(shared_bands.sel(band='green')*(-0.243))+(shared_bands.sel(band='red')*(-0.5424))+(shared_bands.sel(band='nir')*0.7276)+(shared_bands.sel(band='sw1')*0.0713)+(shared_bands.sel(band='sw2')*(-0.1608)))
        # wetness
        shared_indices.append((shared_bands.sel(band='blue')*0.1511)+(shared_bands.sel(band='green')*0.1973)+(shared_bands.sel(band='red')*0.3283)+(shared_bands.sel(band='nir')*0.3407)+(shared_bands.sel(band='sw1')*(-0.7117))+(shared_bands.sel(band='sw2')*(-0.4559)))
        # chlorophyll index green
        shared_indices.append((shared_bands.sel(band='nir')/shared_bands.sel(band='green'))-1)
        #perpendicular impervious surface index  (PISI- isa_index)  	0.8192 * B - 0.5735 * N + 0.0750
        shared_indices.append(0.8192*shared_bands.sel(band='blue')-0.5735*shared_bands.sel(band='nir')+0.0750)
        #carotenoid reflectance index 1
        shared_indices.append((1/shared_bands.sel(band='blue'))-(1/shared_bands.sel(band='green')))
        # chlorophyll vegetation index (N * R) / (G ** 2.0)
        shared_indices.append((shared_bands.sel(band='nir')*shared_bands.sel(band='red'))/(shared_bands.sel(band='green')**2))
        # land surface water index (N - S1)/(N + S1)
        shared_indices.append((shared_bands.sel(band='nir')-shared_bands.sel(band='sw1'))/(shared_bands.sel(band='nir')+shared_bands.sel(band='sw1')))

        # add named band dimension to each array in list
        index_names_list = ['tdvi','ndwi','msavi','brightness','greenness','wetness','chlorophyll_index_green','isa_index','carotenoid_index_1','chlorophyll_veg_index','lswi']
        shared_indices = [x.expand_dims(dim={'band':[index_names_list[i]]}) for i, x in enumerate(shared_indices)]

        # daily time series of shared indices
        shared_indices = xr.concat(shared_indices,dim='band')

        annual_shared, monthly_shared = self.aggregate_annual_monthly(shared_indices)

        sentinel_indices = []

        # chlorophyll index red edge (N/RE1)-1
        sentinel_indices.append((shared_bands.sel(band='nir')/s_only_bands.sel(band='rededge1'))-1)
        # normalized difference red edge index (NDREI) (N - RE1) / (N + RE1)
        sentinel_indices.append((shared_bands.sel(band='nir')-s_only_bands.sel(band='rededge1'))/(shared_bands.sel(band='nir')+s_only_bands.sel(band='rededge1')))
        # inverted red edge chlorophyll index (IRECI) (RE3 - R) / (RE1 / RE2)
        sentinel_indices.append((s_only_bands.sel(band='rededge3')-shared_bands.sel(band='red'))/(s_only_bands.sel(band='rededge1')/s_only_bands.sel(band='rededge2')))
        # carotenoid index 2 (1.0 / B) - (1.0 / RE1)
        sentinel_indices.append((1/shared_bands.sel(band='blue'))-(1/s_only_bands.sel(band='rededge1')))
        #  Anthocyanin Reflectance Index 2   N * ((1 / G) - (1 / RE1))
        sentinel_indices.append(shared_bands.sel(band='nir')*((1/shared_bands.sel(band='green'))-(1/s_only_bands.sel(band='rededge1'))))
        # red edge ndvi 	(RE2 - RE1)/(RE2 + RE1)
        sentinel_indices.append((s_only_bands.sel(band='rededge2')-s_only_bands.sel(band='rededge1'))/(s_only_bands.sel(band='rededge2')+s_only_bands.sel(band='rededge1')))
        # s2 water index (RE1 - S2)/(RE1 + S2)
        sentinel_indices.append((s_only_bands.sel(band='rededge1')-shared_bands.sel(band='sw2'))/((s_only_bands.sel(band='rededge1')+shared_bands.sel(band='sw2'))))

        index_names_list = ['chlorophyll_index_red_edge','ndrei','ireci','carotenoid_index_2','anthocyanin_index_2','red_edge_ndvi','s2_water_index']
        sentinel_indices = [x.expand_dims(dim={'band':[index_names_list[i]]}) for i, x in enumerate(sentinel_indices)]

        # daily time series of sentinel indices
        sentinel_indices = xr.concat(sentinel_indices,dim='band')

        annual_sentinel, monthly_sentinel = self.aggregate_annual_monthly(sentinel_indices)

        ## landsat indices
        # enhanced built up and bareness index (EBBI) (S1 - N) / (10.0 * ((S1 + T) ** 0.5))
        ebbi = (shared_bands.sel(band='sw1')-shared_bands.sel(band='nir'))/(10.0*((shared_bands.sel(band='sw1') + l_only_bands.sel(band='tir1'))**0.5))

        ebbi = ebbi.expand_dims(dim={'band':['ebbi']})

        annual_landsat, monthly_landsat = self.aggregate_annual_monthly(ebbi)

        # stack all annual and monthly time series
        all_annual = xr.concat([annual_shared,annual_sentinel,annual_landsat],dim='band')
        all_monthly = xr.concat([monthly_shared,monthly_sentinel,monthly_landsat],dim='band')

        self.output_indices = xr.concat([all_annual,all_monthly],dim='time')

    def save_outputs(self):
        self.output_bands.to_netcdf(self.site_obj.root / 'data' / self.site_obj.site_name / f'{self.site_obj.site_name}_hls_bands.nc')
        print(f'Bands saved to {self.site_obj.root} / data / {self.site_obj.site_name} / {self.site_obj.site_name}_hls_bands.nc')

        self.output_indices.to_netcdf(self.site_obj.root / 'data' / self.site_obj.site_name / f'{self.site_obj.site_name}_hls_indices.nc')
        print(f'Bands saved to {self.site_obj.root} / data / {self.site_obj.site_name} / {self.site_obj.site_name}_hls_indices.nc')

    def check_outputs(self,type):
        if type == 'indices':
            a = self.output_indices
            if a.sizes['band'] == 19:
                print(f'All indices present')
            else:
                print('Index missing?')
            print(f'Band: {a.band.values}')
        else:
            a = self.output_bands
            if a.sizes['band'] == 12:
                print(f'All bands present')
            else:
                print('Band missing?')
            print(f'Band: {a.band.values}')


### Boston

In [9]:
boston = Site(root=root,site_name='boston', epsg=26919)

In [14]:
b_2016 = HlsDownloader(site_object=boston,year=2016)

In [15]:
b_2016.get_stac_items()

found 26 landsat images
found 43 sentinel images
Suitable landsat images: Index(['2016-04-24', '2016-06-27', '2016-07-13', '2016-08-30', '2016-10-17'], dtype='object')
Suitable sentinel images: Index(['2016-06-17', '2016-06-24', '2016-08-03', '2016-08-23', '2016-09-12',
       '2016-09-25', '2016-10-15', '2016-11-14'],
      dtype='object')


In [ ]:
b_2016.plot_data(satellite='landsat',type='filtered')

In [16]:
b_2016.get_months_to_keep(satellite='landsat',list_of_months=[4,7])

In [17]:
b_2017 = HlsDownloader(site_object=boston,year=2017)
b_2017.get_stac_items()

found 22 landsat images
found 52 sentinel images
Suitable landsat images: Index(['2017-07-16', '2017-08-01', '2017-10-04', '2017-10-20'], dtype='object')
Suitable sentinel images: Index(['2017-06-12', '2017-07-02', '2017-07-09', '2017-08-06', '2017-08-31',
       '2017-10-17'],
      dtype='object')


In [18]:
b_2017.get_months_to_keep(satellite='landsat',list_of_months=[10])

In [19]:
b_2018 = HlsDownloader(site_object=boston,year=2018)
b_2018.get_stac_items()

found 26 landsat images
found 155 sentinel images
Suitable landsat images: Index(['2018-06-17', '2018-07-19', '2018-09-05', '2018-09-21'], dtype='object')
Suitable sentinel images: Index(['2018-04-05', '2018-04-23', '2018-05-08', '2018-05-30', '2018-06-12',
       '2018-06-17', '2018-06-19', '2018-06-29', '2018-07-02', '2018-07-07',
       '2018-07-09', '2018-07-12', '2018-07-19', '2018-08-26', '2018-08-28',
       '2018-09-05', '2018-09-30', '2018-10-17', '2018-10-22', '2018-11-04',
       '2018-11-11'],
      dtype='object')


In [20]:
b_2018.get_months_to_keep(satellite='sentinel',list_of_months=[4,7,10])

In [25]:
boston_combine = DataCombiner(boston,[b_2016,b_2017,b_2018],2016)

In [26]:
boston_combine.combine_data()

In [31]:
boston_combine.save_outputs()

Bands saved to c:\Users\roseh\OneDrive - Hunter - CUNY\Documents\urban_canopy / data / boston / boston_hls_bands.nc
Bands saved to c:\Users\roseh\OneDrive - Hunter - CUNY\Documents\urban_canopy / data / boston / boston_hls_indices.nc


In [ ]:
TC_FILENAME = 'tc_boston_30m.tif'
CITY = 'boston'
BOUNDARY_FILE = 'boston_boundary.gpkg'
FROM_DATE = '2018-04-01'
TO_DATE = '2018-11-30'

In [ ]:
# 2016
l_drop = check_data(landsat_crop,'first')
# manually drop times with partial data
l_drop = l_drop.isel(time=[0,3,4,5,7])

In [ ]:
# 2018
#l_sept = check_data(landsat_crop,'first')
l_sept = l_sept.isel(time=slice(2,4))
l_sept

In [102]:
# 2020
#l_nov = check_data(landsat_crop,'last')
l_nov = l_nov.isel(time=4)
l_nov = l_nov.expand_dims({'time':[l_nov.coords['time'].values]})

In [ ]:
# 2020
l_may = check_data(landsat_crop,'first')  # first and last refer to which duplicate to drop
l_may = l_may.isel(time=slice(1,3))
l_may

In [ ]:
s_drop = check_data(sentinel_crop,'first')  

# manually drop times with partial data
to_drop = s_drop.time.values[[3,5,12,15,20,21,28,31,33]]
s_drop = s_drop.drop_sel({'time':to_drop})

In [103]:
# save image dates to csv
all_landsat_times = np.concat([l_drop.time.values, l_sept.time.values,l_may.time.values,l_nov.time.values])

sdf = pd.DataFrame({'sentinel':s_drop.time.values})
ldf = pd.DataFrame({'landsat':all_landsat_times})       

new = pd.concat([sdf, ldf], axis=1) 
new.to_csv(root / 'output' / CITY / f'{CITY}_satellite_images.csv')

In [ ]:
# change landsat dates to same year and combine

# nov and may from 2020
l_nov['time'] = l_nov.time.to_index() - pd.Timedelta(days=365*4)
l_may['time'] = l_may.time.to_index() - pd.Timedelta(days=365*4)
# september from 2018
l_sept['time'] = l_sept.time.to_index() - pd.Timedelta(days=365*2)

l_combine = l_drop.combine_first(l_may).combine_first(l_sept).combine_first(l_nov)

### Cloud Mask and Scale

In [111]:
l_masked = mask_and_scale2(l_combine, sensor='landsat',scale_factor1= .0001, scale_factor2= .01)
s_masked = mask_and_scale2(s_drop,sensor='sentinel',scale_factor1= .0001, scale_factor2=None)

In [113]:
# dates need to be chanaged to the same year in order for merge to work

# sentinel from 2018
s_masked['time'] = s_masked.time.to_index() - pd.Timedelta(days=365*2)


### Calculate vegetation indices, merge landsat and sentinel data

In [115]:
all_unique_indices, all_annual_unique_indices = get_unique_indices(l_data=l_masked,s_data=s_masked)
all_shared_indices, all_annual_shared_indices = get_shared_indices(l_data=l_masked,s_data=s_masked)

t = concatenate_and_save_data(annual1=all_annual_unique_indices,annual2=all_annual_shared_indices,
                              monthly1=all_unique_indices,monthly2=all_shared_indices,root=root,filename=f'{CITY}_hls_indices2')

c:\Users\roseh\miniconda3\envs\rs-env\Lib\site-packages\dask\array\core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 11
  result = blockwise(
c:\Users\roseh\miniconda3\envs\rs-env\Lib\site-packages\dask\array\core.py:4888: PerformanceWarning: Increasing number of chunks by factor of 11
  result = blockwise(


In [116]:
t

<xarray.DataArray 'stackstac-9564630fb352892a4135dadb186a6f8b' (band: 19,
                                                                time: 9,
                                                                y: 642, x: 550)> Size: 483MB
dask.array<concatenate, shape=(19, 9, 642, 550), dtype=float64, chunksize=(1, 1, 642, 550), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) object 152B 'chlorophyll_index_red_edge' ... 'lswi'
  * x            (x) float64 4kB 8.141e+05 8.141e+05 ... 8.305e+05 8.306e+05
  * y            (y) float64 5kB 4.702e+06 4.702e+06 ... 4.682e+06 4.682e+06
    epsg         int64 8B 26918
    spatial_ref  int64 8B 0
  * time         (time) <U9 324B 'april' 'may' 'june' ... 'november' 'annual'

In [117]:
all_unique_bands, all_annual_unique = get_unique_bands(l_data=l_masked,s_data=s_masked)
all_shared_bands, all_annual_shared = get_shared_bands(landsat_data=l_masked,sentinel_data=s_masked)


s = concatenate_and_save_data(annual1=all_annual_shared,annual2=all_annual_unique,
                              monthly1=all_unique_bands,monthly2=all_shared_bands, root=root,filename=f'{CITY}_hls_bands2',
                              # extra arguments needed for adding tree canopy layer
                              add_canopy=True,shoreline=shoreline,tc_filename=TC_FILENAME,city_boundary=boundary)


### Save as Rasters

In [ ]:
# def save_monthly_rasters(data,filename):
#     months = ['april','may','june','july','august','september','october','november']
#     for i in range(0,8):
#         data.isel(time=i).rio.to_raster(root / 'data' / f'{filename}_{months[i]}.tif')


# all_annual.isel(time=0).rio.to_raster(root / 'data' / 'annual_bands.tif')     

# save_monthly_rasters(all_shared_bands,filename='shared_bands')
# save_monthly_rasters(all_unique_bands,filename='unique_bands')
# save_monthly_rasters(all_shared_indices,filename='shared_indices')